# Lab 07 — Ontology 用のテーブルを整備

Ontology の各エンティティに対応するテーブルを、加工済みデータから作成します。

| エンティティ | テーブル | キー |
|---|---|---|
| Trip | `onto_trips` | `trip_id` |
| Zone | `onto_zones` | `LocationID` |
| Vendor | `onto_vendors` | `VendorID` |

`onto_trips` は 1 トリップ 1 行のまま、支払方法・乗車種別の名前を付けます。件数・金額は変えません。

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
from pyspark.sql import functions as F

trips = spark.table("green_tripdata_2026")
onto_trips = (
    trips.join(spark.table("payment_type_lookup"), "payment_type", "left")
    .join(spark.table("trip_type_lookup"), "trip_type", "left")
    .select(
        F.col("trip_id").cast("long"), F.col("VendorID").cast("long"), F.col("PULocationID").cast("long"),
        F.col("DOLocationID").cast("long"), "lpep_pickup_datetime", F.col("pickup_date").cast("timestamp").alias("pickup_date"),
        F.col("pickup_hour").cast("long"), F.col("passenger_count").cast("long"), "passenger_group", "trip_distance",
        "trip_duration_minutes", "fare_amount", "tip_amount", "total_amount", "payment_type_name", "trip_type_name",
    )
)
onto_zones = spark.table("taxi_zone_lookup").select(F.col("LocationID").cast("long"), "Borough", "Zone", "service_zone")
onto_vendors = spark.table("vendor_lookup").select(F.col("VendorID").cast("long"), "vendor_name")

for name, frame in {"onto_trips": onto_trips, "onto_zones": onto_zones, "onto_vendors": onto_vendors}.items():
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)

for name, rows in {"onto_trips": 37948, "onto_zones": 265, "onto_vendors": 2}.items():
    check(f"{name} rows", rows, spark.table(name).count())
for name, column in {"onto_trips": "trip_id", "onto_zones": "LocationID", "onto_vendors": "VendorID"}.items():
    check(f"{name} key unique", spark.table(name).count(), spark.table(name).select(column).distinct().count())
check("onto_trips total_amount unchanged", float(trips.agg(F.round(F.sum("total_amount"), 2)).first()[0]),
      float(spark.table("onto_trips").agg(F.round(F.sum("total_amount"), 2)).first()[0]), 0.005)
check("onto_trips names resolved", 0, spark.table("onto_trips").filter(
    F.col("payment_type_name").isNull() | F.col("trip_type_name").isNull()).count())

failed = [c for c in checks if not c["passed"]]
save_result("nb_07_prepare_ontology", {"passed": not failed, "failed": len(failed), "checks": checks})